
# NDJF Cluster Composites and Representative Events

This notebook turns the validated `k = 3` working subtype framework into full-domain physical composite maps.

Composite rule:
- these are **not** composites of the cluster variables themselves
- these are composites of the full gridded physical fields used to calculate the cluster-defining event summaries
- the computations are done across the **entire objective-subtype domain**, not only inside the characterization boxes

Primary composite stage:
- use `Cluster 1`, `Cluster 2`, and `Cluster 3` from the validated `k = 3` solution
- ignore the `k = 4` solution for the main composite set
- build `9` primary full-grid composites = `3` physical fields times `3` clusters

Primary composite fields:
- `925 hPa` convergence at the event peak time
- `850 hPa` geopotential-height anomaly minimum over `t-12`, `t0`, `t+12`
- `850 hPa` temperature-gradient magnitude maximum over `t-12`, `t0`, `t+12`

Supporting outputs:
- pointwise sample-count maps
- pointwise standard-deviation maps
- event-level and cluster-level box-average summaries
- optional cluster-difference maps

Display note:
- the main interpretation maps intentionally omit characterization-box overlays so the eye can focus on whether the cluster structures actually differ
- the box summaries are still saved separately for quantitative comparison

Zero and missing-data rule used here:
- include zeros when they are real physical values
- exclude only missing values from the numerators and denominators
- use that same rule consistently across every composite process


In [ ]:

import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = True
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:

from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
import xarray as xr

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import (
    COASTAL_JAPAN_BOX,
    HOKKAIDO_BOX,
    HOKKAIDO_FRONT_BOX,
    JPCZ_POLYGON_VERTICES,
    OBJECTIVE_SUBTYPE_DOMAIN,
    PACIFIC_EAST_OF_JAPAN_BOX,
    PACIFIC_FRONT_BOX,
    SEA_OF_JAPAN_BOX,
)
from jpcz_catalog.detect import compute_divergence_field, prepare_detection_geometry
from jpcz_catalog.diagnostics import (
    compute_geopotential_height_field,
    compute_temperature_gradient_magnitude,
    load_offset_snapshot,
    load_snapshot,
)
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.satellite import default_modis_layers_for_date
from jpcz_catalog.subtypes import feature_definitions_dataframe, standardize_feature_table

RUN_EXPORT_DIR_08 = Path("outputs/verification/objective_subtype_runs")
RUN_EXPORT_DIR_09 = Path("outputs/verification/objective_subtype_validation")
COMPOSITE_EXPORT_DIR = Path("outputs/verification/objective_subtype_cluster_examples")
COMPOSITE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_subtype_cluster_example_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
CLIMATOLOGY_PATH = Path("outputs/verification/z850_ndjf_monthly_climatology.nc")
FEATURE_TABLE_PATH = Path("outputs/verification/jpcz_catalog_ndjf_objective_subtype_features.csv")

QUICKLOOK_DIR = Path("outputs/verification/ndjf_event_quicklooks_merged_12h")
SATELLITE_DIR = Path("outputs/verification/ndjf_event_satellite_panels_merged_12h")
OLR_DIR = Path("outputs/verification/ndjf_event_olr_panels_merged_12h")

PRIMARY_CLUSTER_K = 3
PRIMARY_CLUSTER_COLUMN = f"cluster_ward_{PRIMARY_CLUSTER_K}"
REPRESENTATIVE_EVENTS_PER_CLUSTER = 5
DISPLAY_REPRESENTATIVE_EVENTS_PER_CLUSTER = 1
SAVE_PLOTS = True
FORCE_REBUILD_CLUSTER_COMPOSITES = False
ERA5_TIME_CHUNK = 48
PROGRESS_EVERY = 10
SYNOPTIC_OFFSETS = (-12, 0, 12)
MIN_FRACTION_EVENTS_TO_PLOT = 0.8

PRIMARY_COMPOSITE_FIELDS = [
    "convergence_925_peak",
    "z850_anomaly_min_tminus12_to_tplus12",
    "temperature_gradient_850_max_tminus12_to_tplus12",
]

CLUSTER_FEATURE_COLUMNS = [
    "coastal_to_jpcz_mean_convergence_ratio",
    "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
    "front_box_max_temp_gradient_850_tminus12_to_tplus12",
    "sea_of_japan_mean_vorticity_peak_925",
]

CLUSTER_LABELS = {
    1: "Cluster 1: least synoptic / weaker-background",
    2: "Cluster 2: moderately synoptic / circulation-modified",
    3: "Cluster 3: strongly synoptic / frontal / coastal-enhanced",
}

BOXES = {
    "JPCZ polygon envelope": None,
    "Coastal Japan": COASTAL_JAPAN_BOX,
    "Pacific east of Japan": PACIFIC_EAST_OF_JAPAN_BOX,
    "Hokkaido": HOKKAIDO_BOX,
    "Sea of Japan": SEA_OF_JAPAN_BOX,
    "Hokkaido front": HOKKAIDO_FRONT_BOX,
    "Pacific front": PACIFIC_FRONT_BOX,
}

FEATURE_DICTIONARY = feature_definitions_dataframe()
FEATURE_UNITS = FEATURE_DICTIONARY.set_index("column_name")["units"].to_dict()

COMPOSITE_MEAN_PATH = COMPOSITE_EXPORT_DIR / "k3_full_domain_cluster_composite_means.nc"
COMPOSITE_STD_PATH = COMPOSITE_EXPORT_DIR / "k3_full_domain_cluster_composite_std.nc"
COMPOSITE_COUNT_PATH = COMPOSITE_EXPORT_DIR / "k3_full_domain_cluster_composite_counts.nc"
COMPOSITE_EVENT_BOX_PATH = COMPOSITE_EXPORT_DIR / "k3_event_level_box_means.csv"
COMPOSITE_BOX_SUMMARY_PATH = COMPOSITE_EXPORT_DIR / "k3_cluster_box_mean_summary.csv"
COMPOSITE_MAP_BOX_PATH = COMPOSITE_EXPORT_DIR / "k3_cluster_composite_map_box_means.csv"
COMPOSITE_DIFF_PATH = COMPOSITE_EXPORT_DIR / "k3_cluster_difference_maps.nc"
PARTIAL_STATUS_PATH = COMPOSITE_EXPORT_DIR / "k3_full_domain_cluster_composite_partial_status.csv"


def maybe_copy_to_drive(path: Path):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if path.is_file():
        shutil.copy2(path, drive_path)
        print("Copied to Drive:", drive_path)



def restore_from_drive_cache(path: Path) -> bool:
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if not drive_path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True



def axis_label(column_name: str) -> str:
    units = FEATURE_UNITS.get(column_name)
    if units is None or units == "unitless":
        return column_name
    return f"{column_name}\n[{units}]"



def quicklook_name_for_row(row_index: int, row: pd.Series) -> str:
    return f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{row_index:03d}.png"



def satellite_name_for_row(row_index: int, row: pd.Series) -> str | None:
    satellite_layers = default_modis_layers_for_date(pd.Timestamp(row['event_peak']).normalize())
    if not satellite_layers:
        return None
    satellite_layer = satellite_layers[0]
    layer_slug = (
        satellite_layer.replace("MODIS_", "")
        .replace("_CorrectedReflectance_TrueColor", "")
        .lower()
    )
    return f"{pd.Timestamp(row['event_peak']).strftime('%Y%m%dT%H%M')}_idx{row_index:03d}_{layer_slug}.jpg"



def ensure_local_copy(local_path: Path, drive_subdir_name: str) -> bool:
    if local_path.exists():
        return True
    drive_path = Path(DRIVE_OUTPUT_DIR) / drive_subdir_name / local_path.name
    if not drive_path.exists():
        return False
    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, local_path)
    return True



def lat_weighted_box_mean(field: xr.DataArray, box) -> float:
    subset = field.sel(
        longitude=slice(box.lon_min, box.lon_max),
        latitude=slice(box.lat_max, box.lat_min),
    )
    if subset.size == 0:
        return float("nan")
    lat_weights = xr.DataArray(
        np.cos(np.deg2rad(subset.latitude.values)),
        coords={"latitude": subset.latitude.values},
        dims=("latitude",),
    ).broadcast_like(subset)
    valid = xr.apply_ufunc(np.isfinite, subset)
    numerator = (subset.where(valid, 0.0) * lat_weights.where(valid, 0.0)).sum()
    denominator = lat_weights.where(valid, 0.0).sum()
    denominator_value = float(denominator.values)
    if denominator_value <= 0.0:
        return float("nan")
    return float((numerator / denominator).values)



def initialize_accumulator_dataset(event_ds: xr.Dataset) -> xr.Dataset:
    data_vars = {}
    for field_name in event_ds.data_vars:
        field = event_ds[field_name]
        valid = xr.apply_ufunc(np.isfinite, field)
        filled = field.where(valid, 0.0)
        data_vars[f"sum__{field_name}"] = filled.astype(float)
        data_vars[f"sumsq__{field_name}"] = (filled ** 2).astype(float)
        data_vars[f"count__{field_name}"] = valid.astype(np.int32)
    return xr.Dataset(data_vars)



def update_accumulator_dataset(accumulator_ds: xr.Dataset, event_ds: xr.Dataset) -> xr.Dataset:
    updates = {}
    for field_name in event_ds.data_vars:
        field = event_ds[field_name]
        valid = xr.apply_ufunc(np.isfinite, field)
        filled = field.where(valid, 0.0)
        updates[f"sum__{field_name}"] = accumulator_ds[f"sum__{field_name}"] + filled
        updates[f"sumsq__{field_name}"] = accumulator_ds[f"sumsq__{field_name}"] + (filled ** 2)
        updates[f"count__{field_name}"] = accumulator_ds[f"count__{field_name}"] + valid.astype(np.int32)
    return xr.Dataset(updates)



def finalize_accumulator_dataset(accumulator_ds: xr.Dataset) -> tuple[xr.Dataset, xr.Dataset, xr.Dataset]:
    mean_vars = {}
    std_vars = {}
    count_vars = {}
    for data_var in accumulator_ds.data_vars:
        if not data_var.startswith("sum__"):
            continue
        field_name = data_var.replace("sum__", "", 1)
        sums = accumulator_ds[f"sum__{field_name}"]
        sumsq = accumulator_ds[f"sumsq__{field_name}"]
        counts = accumulator_ds[f"count__{field_name}"]
        mean_field = xr.where(counts > 0, sums / counts, np.nan).rename(field_name)
        variance_field = xr.where(counts > 1, (sumsq / counts) - (mean_field ** 2), np.nan)
        variance_field = xr.where(variance_field < 0, 0.0, variance_field)
        std_field = xr.where(counts > 1, variance_field ** 0.5, np.nan).rename(field_name)
        mean_vars[field_name] = mean_field
        std_vars[field_name] = std_field
        count_vars[field_name] = counts.rename(field_name)
    return xr.Dataset(mean_vars), xr.Dataset(std_vars), xr.Dataset(count_vars)



def write_partial_status(status_lookup: dict[int, dict]):
    partial_status_df = pd.DataFrame([status_lookup[k] for k in sorted(status_lookup)])
    partial_status_df.to_csv(PARTIAL_STATUS_PATH, index=False)
    maybe_copy_to_drive(PARTIAL_STATUS_PATH)



def write_partial_accumulator_checkpoint(
    accumulator_ds: xr.Dataset,
    *,
    cluster_id: int,
    processed_events: int,
    total_events: int,
    partial_path: Path,
):
    checkpoint_ds = accumulator_ds.copy(deep=True)
    checkpoint_ds.attrs["cluster_id"] = int(cluster_id)
    checkpoint_ds.attrs["cluster_label"] = CLUSTER_LABELS[int(cluster_id)]
    checkpoint_ds.attrs["processed_events"] = int(processed_events)
    checkpoint_ds.attrs["total_events"] = int(total_events)
    checkpoint_ds.to_netcdf(partial_path)
    maybe_copy_to_drive(partial_path)


In [ ]:

paths_to_restore = [
    FEATURE_TABLE_PATH,
    CLIMATOLOGY_PATH,
    RUN_EXPORT_DIR_08 / "clustered_events_k3.csv",
    RUN_EXPORT_DIR_08 / "cluster_medians_k3.csv",
    RUN_EXPORT_DIR_08 / "cluster_counts_k3.csv",
    RUN_EXPORT_DIR_08 / "cluster_comparison_summary.csv",
    RUN_EXPORT_DIR_09 / "cluster_validation_summary.csv",
    RUN_EXPORT_DIR_09 / "cluster_nesting_k2_vs_k3_counts.csv",
    RUN_EXPORT_DIR_09 / "cluster_nesting_k2_vs_k3_row_fraction.csv",
    RUN_EXPORT_DIR_09 / "pairwise_cluster_checks_k3.csv",
]
for path in paths_to_restore:
    if not path.exists():
        restore_from_drive_cache(path)

clustered_k3_df = pd.read_csv(RUN_EXPORT_DIR_08 / "clustered_events_k3.csv")
clustered_k3_df = add_japan_local_time_columns(clustered_k3_df)
if PRIMARY_CLUSTER_COLUMN not in clustered_k3_df.columns:
    cluster_cols = [c for c in clustered_k3_df.columns if c.startswith("cluster_")]
    raise RuntimeError(f"Expected {PRIMARY_CLUSTER_COLUMN} in clustered_events_k3.csv, found {cluster_cols}")

cluster_counts_path = RUN_EXPORT_DIR_08 / "cluster_counts_k3.csv"
cluster_medians_path = RUN_EXPORT_DIR_08 / "cluster_medians_k3.csv"
if cluster_counts_path.exists():
    cluster_counts_df = pd.read_csv(cluster_counts_path, index_col=0)
else:
    cluster_counts_df = clustered_k3_df[PRIMARY_CLUSTER_COLUMN].value_counts().sort_index().rename("n_events").to_frame()
if cluster_medians_path.exists():
    cluster_medians_df = pd.read_csv(cluster_medians_path, index_col=0)
else:
    cluster_medians_df = clustered_k3_df.groupby(PRIMARY_CLUSTER_COLUMN).median(numeric_only=True).round(2)
validation_summary_df = pd.read_csv(RUN_EXPORT_DIR_09 / "cluster_validation_summary.csv")
k2_vs_k3_counts_df = pd.read_csv(RUN_EXPORT_DIR_09 / "cluster_nesting_k2_vs_k3_counts.csv", index_col=0)
k2_vs_k3_fraction_df = pd.read_csv(RUN_EXPORT_DIR_09 / "cluster_nesting_k2_vs_k3_row_fraction.csv", index_col=0)
pairwise_k3_df = pd.read_csv(RUN_EXPORT_DIR_09 / "pairwise_cluster_checks_k3.csv")

standardized_df, feature_means, feature_stds = standardize_feature_table(
    clustered_k3_df.copy(),
    columns=CLUSTER_FEATURE_COLUMNS,
)
cluster_labels = clustered_k3_df[PRIMARY_CLUSTER_COLUMN]
valid_mask = standardized_df.notna().all(axis=1) & cluster_labels.notna()
standardized_valid = standardized_df.loc[valid_mask].copy()
labels_valid = cluster_labels.loc[valid_mask].astype(int)
centroids = standardized_valid.groupby(labels_valid).mean()

representative_records = []
for cluster_id in sorted(centroids.index):
    member_index = labels_valid.index[labels_valid == cluster_id]
    member_points = standardized_valid.loc[member_index]
    centroid = centroids.loc[cluster_id]
    distances = np.sqrt(((member_points - centroid) ** 2).sum(axis=1))
    top_members = distances.sort_values().head(REPRESENTATIVE_EVENTS_PER_CLUSTER)

    for rank_within_cluster, (row_index, centroid_distance) in enumerate(top_members.items(), start=1):
        row = clustered_k3_df.loc[row_index].copy()
        quicklook_name = quicklook_name_for_row(row_index, row)
        satellite_name = satellite_name_for_row(row_index, row)
        olr_name = quicklook_name
        quicklook_exists = ((QUICKLOOK_DIR / quicklook_name).exists() or (Path(DRIVE_OUTPUT_DIR) / QUICKLOOK_DIR.name / quicklook_name).exists())
        olr_exists = ((OLR_DIR / olr_name).exists() or (Path(DRIVE_OUTPUT_DIR) / OLR_DIR.name / olr_name).exists())
        satellite_exists = False if satellite_name is None else ((SATELLITE_DIR / satellite_name).exists() or (Path(DRIVE_OUTPUT_DIR) / SATELLITE_DIR.name / satellite_name).exists())
        representative_records.append(
            {
                "cluster_id": int(cluster_id),
                "cluster_label": CLUSTER_LABELS.get(int(cluster_id), f"Cluster {int(cluster_id)}"),
                "representative_rank": int(rank_within_cluster),
                "catalog_index": int(row_index),
                "centroid_distance": float(centroid_distance),
                "event_peak_utc": pd.Timestamp(row["event_peak"]),
                "event_peak_jst": pd.Timestamp(row["event_peak_jst"]) if pd.notna(row.get("event_peak_jst")) else pd.NaT,
                "duration_hours": float(row["duration_hours"]),
                "event_peak_D_1e5_s-1": float(row["event_peak_D_1e5_s-1"]),
                "coastal_to_jpcz_mean_convergence_ratio": float(row["coastal_to_jpcz_mean_convergence_ratio"]),
                "sea_of_japan_mean_vorticity_peak_925": float(row["sea_of_japan_mean_vorticity_peak_925"]),
                "hokkaido_min_z850_anomaly_tminus12_to_tplus12": float(row["hokkaido_min_z850_anomaly_tminus12_to_tplus12"]),
                "front_box_max_temp_gradient_850_tminus12_to_tplus12": float(row["front_box_max_temp_gradient_850_tminus12_to_tplus12"]),
                "pattern_type_manual": row.get("pattern_type_manual", ""),
                "verification_notes": row.get("verification_notes", ""),
                "quicklook_name": quicklook_name,
                "olr_name": olr_name,
                "satellite_name": satellite_name,
                "quicklook_exists": bool(quicklook_exists),
                "olr_exists": bool(olr_exists),
                "satellite_exists": bool(satellite_exists),
            }
        )

representative_events_df = pd.DataFrame(representative_records).sort_values(["cluster_id", "representative_rank"]).reset_index(drop=True)
representatives_path = COMPOSITE_EXPORT_DIR / "k3_representative_events.csv"
representative_events_df.to_csv(representatives_path, index=False)
maybe_copy_to_drive(representatives_path)

print("Loaded k=3 working subtype outputs")
display(cluster_counts_df)
print("\nValidation summary row for k=3")
display(validation_summary_df.loc[validation_summary_df["k"] == 3])
print("\nk=2 vs k=3 nesting counts")
display(k2_vs_k3_counts_df)
print("\nk=2 vs k=3 nesting row fractions")
display(k2_vs_k3_fraction_df)
print("\nRepresentative events nearest the k=3 cluster centroids")
display(representative_events_df)


In [ ]:

def display_representative_panels(representative_df: pd.DataFrame, *, per_cluster: int = 1):
    rows_to_show = representative_df.groupby("cluster_id", group_keys=False).head(per_cluster)
    for _, rep in rows_to_show.iterrows():
        print(rep["cluster_label"])
        print(
            f"catalog idx {int(rep['catalog_index'])} | UTC {pd.Timestamp(rep['event_peak_utc']):%Y-%m-%d %H:%M} | "
            f"JST {pd.Timestamp(rep['event_peak_jst']):%Y-%m-%d %H:%M} | centroid distance {rep['centroid_distance']:.3f}"
        )

        panel_specs = [
            ("Convergence quicklook", QUICKLOOK_DIR / rep["quicklook_name"], QUICKLOOK_DIR.name),
            ("OLR panel", OLR_DIR / rep["olr_name"], OLR_DIR.name),
        ]
        if pd.notna(rep["satellite_name"]) and rep["satellite_name"]:
            panel_specs.append(("Satellite panel", SATELLITE_DIR / rep["satellite_name"], SATELLITE_DIR.name))

        fig, axes = plt.subplots(1, len(panel_specs), figsize=(6 * len(panel_specs), 6))
        if len(panel_specs) == 1:
            axes = [axes]

        for ax, (panel_label, local_path, drive_subdir_name) in zip(axes, panel_specs):
            has_panel = ensure_local_copy(local_path, drive_subdir_name)
            if not has_panel:
                ax.axis("off")
                ax.set_title(f"{panel_label}\nmissing")
                continue
            image = mpimg.imread(local_path)
            ax.imshow(image)
            ax.axis("off")
            ax.set_title(panel_label)

        fig.suptitle(rep["cluster_label"], y=0.98)
        fig.tight_layout()
        plt.show()


display_representative_panels(
    representative_events_df,
    per_cluster=DISPLAY_REPRESENTATIVE_EVENTS_PER_CLUSTER,
)


In [ ]:

if not CLIMATOLOGY_PATH.exists():
    restore_from_drive_cache(CLIMATOLOGY_PATH)
z850_climatology = xr.open_dataarray(CLIMATOLOGY_PATH).load()

for path in [
    COMPOSITE_MEAN_PATH,
    COMPOSITE_STD_PATH,
    COMPOSITE_COUNT_PATH,
    COMPOSITE_EVENT_BOX_PATH,
    COMPOSITE_BOX_SUMMARY_PATH,
    COMPOSITE_MAP_BOX_PATH,
    COMPOSITE_DIFF_PATH,
    PARTIAL_STATUS_PATH,
]:
    if not path.exists():
        restore_from_drive_cache(path)

cluster_ids = sorted(clustered_k3_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).unique())
cluster_event_counts = (
    clustered_k3_df[PRIMARY_CLUSTER_COLUMN].dropna().astype(int).value_counts().sort_index().to_dict()
)
cluster_meaning_df = pd.DataFrame(
    [
        {
            "cluster_id": int(cluster_id),
            "cluster_label": CLUSTER_LABELS[int(cluster_id)],
            "n_events": int(cluster_event_counts[int(cluster_id)]),
        }
        for cluster_id in cluster_ids
    ]
)
cluster_evidence_specs = [
    {
        "feature": "hokkaido_min_z850_anomaly_tminus12_to_tplus12",
        "display_name": "Hokkaido minimum z850 anomaly over t-12/t0/t+12",
        "units": "gpm",
        "interpretation": "More negative values mean a deeper low/trough signal near Hokkaido.",
    },
    {
        "feature": "sea_of_japan_mean_vorticity_peak_925",
        "display_name": "Sea of Japan mean 925 hPa vorticity at peak",
        "units": "1e-5 s^-1",
        "interpretation": "Larger positive values mean stronger cyclonic Sea of Japan circulation.",
    },
    {
        "feature": "coastal_to_jpcz_mean_convergence_ratio",
        "display_name": "Coastal-to-JPCZ mean convergence ratio at peak",
        "units": "unitless",
        "interpretation": "Smaller values mean more polygon-centered; larger values mean more coastal-enhanced.",
    },
    {
        "feature": "front_box_max_temp_gradient_850_tminus12_to_tplus12",
        "display_name": "Front-box maximum |grad T850| over t-12/t0/t+12",
        "units": "K (100 km)^-1",
        "interpretation": "Larger values mean stronger frontality/baroclinicity.",
    },
]
cluster_medians_lookup = cluster_medians_df.copy()
cluster_medians_lookup.index = pd.Index([int(x) for x in cluster_medians_lookup.index], name="cluster_id")
cluster_evidence_rows = []
for spec in cluster_evidence_specs:
    feature = spec["feature"]
    cluster_evidence_rows.append(
        {
            "feature": feature,
            "display_name": spec["display_name"],
            "units": spec["units"],
            "cluster_1_median": float(cluster_medians_lookup.loc[1, feature]),
            "cluster_2_median": float(cluster_medians_lookup.loc[2, feature]),
            "cluster_3_median": float(cluster_medians_lookup.loc[3, feature]),
            "interpretation": spec["interpretation"],
        }
    )
cluster_evidence_df = pd.DataFrame(cluster_evidence_rows)
for column_name in ["cluster_1_median", "cluster_2_median", "cluster_3_median"]:
    cluster_evidence_df[column_name] = cluster_evidence_df[column_name].round(2)
cluster_evidence_path = COMPOSITE_EXPORT_DIR / "k3_cluster_label_evidence_table.csv"
cluster_evidence_df.to_csv(cluster_evidence_path, index=False)
maybe_copy_to_drive(cluster_evidence_path)
field_display_names = {
    "convergence_925_peak": "925 hPa convergence at event peak",
    "z850_anomaly_min_tminus12_to_tplus12": "850 hPa z anomaly minimum over t-12/t0/t+12",
    "temperature_gradient_850_max_tminus12_to_tplus12": "850 hPa |grad T| maximum over t-12/t0/t+12",
}
composite_field_definition_df = pd.DataFrame(
    [
        {
            "field": "convergence_925_peak",
            "label": "925 hPa convergence at event peak",
            "formula": "1e5 * max(-(du/dx + dv/dy), 0)",
            "time_reduction": "event peak only (t0)",
            "source_fields": "925 hPa u and v winds",
            "units": "1e-5 s^-1",
        },
        {
            "field": "z850_anomaly_min_tminus12_to_tplus12",
            "label": "850 hPa z anomaly minimum over t-12/t0/t+12",
            "formula": "min[z850_event(t) - z850_monthly_climatology(month(t))] over t in {-12,0,+12}",
            "time_reduction": "minimum over t-12, t0, t+12",
            "source_fields": "850 hPa geopotential height",
            "units": "gpm",
        },
        {
            "field": "temperature_gradient_850_max_tminus12_to_tplus12",
            "label": "850 hPa |grad T| maximum over t-12/t0/t+12",
            "formula": "max[1e5 * sqrt((dT/dx)^2 + (dT/dy)^2)] over t in {-12,0,+12}",
            "time_reduction": "maximum over t-12, t0, t+12",
            "source_fields": "850 hPa temperature",
            "units": "K (100 km)^-1",
        },
    ]
)
climatology_years = sorted(pd.to_datetime(clustered_k3_df["event_peak"]).dt.year.unique().tolist())
climatology_summary_df = (
    z850_climatology.mean(dim=("latitude", "longitude"))
    .to_series()
    .rename("domain_mean_gpm")
    .reset_index()
)

print("Composite field definitions used in Notebook 10")
display(composite_field_definition_df)
print("\nValidated k=3 cluster meanings used in the composite figures")
display(cluster_meaning_df)
print("\nEvidence table behind the k=3 cluster labels (medians of the clustering features)")
display(cluster_evidence_df)
print(
    f"\nZ850 monthly climatology loaded from {CLIMATOLOGY_PATH.name} using the study-period event years represented in the k=3 catalog: "
    f"{climatology_years[0]}-{climatology_years[-1]}"
)
display(climatology_summary_df)

def compute_event_primary_fields(ds: xr.Dataset, row: pd.Series, geometry_925=None):
    peak_time = pd.Timestamp(row["event_peak"])
    peak_snapshot_925 = load_snapshot(
        ds,
        peak_time,
        variables=("u_component_of_wind", "v_component_of_wind"),
        domain=OBJECTIVE_SUBTYPE_DOMAIN,
        level=925,
    )
    if geometry_925 is None:
        geometry_925 = prepare_detection_geometry(
            peak_snapshot_925.longitude,
            peak_snapshot_925.latitude,
            JPCZ_POLYGON_VERTICES,
        )

    divergence_field = compute_divergence_field(
        peak_snapshot_925,
        dx=geometry_925.dx,
        dy=geometry_925.dy,
    )
    convergence_field = ((-divergence_field).clip(min=0.0) * 1e5).rename("convergence_925_peak")

    z850_anomaly_stack = []
    temp_gradient_stack = []
    for offset in SYNOPTIC_OFFSETS:
        synoptic_time = peak_time + pd.Timedelta(hours=offset)
        synoptic_snapshot_850 = load_offset_snapshot(
            ds,
            peak_time,
            offset_hours=offset,
            variables=("geopotential", "temperature"),
            domain=OBJECTIVE_SUBTYPE_DOMAIN,
            level=850,
        )
        z850 = compute_geopotential_height_field(synoptic_snapshot_850)
        z850_anomaly = (z850 - z850_climatology.sel(month=synoptic_time.month)).expand_dims(offset_hours=[offset])
        z850_anomaly_stack.append(z850_anomaly)

        temp_gradient = compute_temperature_gradient_magnitude(synoptic_snapshot_850)
        temp_gradient_display = (
            temp_gradient * float(temp_gradient.attrs.get("display_scale_factor", 1.0))
        ).expand_dims(offset_hours=[offset])
        temp_gradient_stack.append(temp_gradient_display)

    z850_anomaly_min = xr.concat(z850_anomaly_stack, dim="offset_hours").min(dim="offset_hours", skipna=True)
    z850_anomaly_min = z850_anomaly_min.rename("z850_anomaly_min_tminus12_to_tplus12")

    temp_gradient_max = xr.concat(temp_gradient_stack, dim="offset_hours").max(dim="offset_hours", skipna=True)
    temp_gradient_max = temp_gradient_max.rename("temperature_gradient_850_max_tminus12_to_tplus12")

    event_ds = xr.Dataset(
        {
            "convergence_925_peak": convergence_field,
            "z850_anomaly_min_tminus12_to_tplus12": z850_anomaly_min,
            "temperature_gradient_850_max_tminus12_to_tplus12": temp_gradient_max,
        }
    )
    return event_ds, geometry_925


if all(path.exists() for path in [COMPOSITE_MEAN_PATH, COMPOSITE_STD_PATH, COMPOSITE_COUNT_PATH, COMPOSITE_EVENT_BOX_PATH, COMPOSITE_BOX_SUMMARY_PATH, COMPOSITE_MAP_BOX_PATH, COMPOSITE_DIFF_PATH]) and not FORCE_REBUILD_CLUSTER_COMPOSITES:
    composite_mean_ds = xr.open_dataset(COMPOSITE_MEAN_PATH).load()
    composite_std_ds = xr.open_dataset(COMPOSITE_STD_PATH).load()
    composite_count_ds = xr.open_dataset(COMPOSITE_COUNT_PATH).load()
    composite_diff_ds = xr.open_dataset(COMPOSITE_DIFF_PATH).load()
    event_box_df = pd.read_csv(COMPOSITE_EVENT_BOX_PATH)
    cluster_box_summary_df = pd.read_csv(COMPOSITE_BOX_SUMMARY_PATH)
    composite_map_box_df = pd.read_csv(COMPOSITE_MAP_BOX_PATH)
    print("Loaded cached full-domain k=3 composite products")
else:
    ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})
    geometry_925 = None
    cluster_mean_datasets = []
    cluster_std_datasets = []
    cluster_count_datasets = []
    event_box_records_all = []
    partial_status_lookup = {}

    for cluster_id in cluster_ids:
        cluster_members = clustered_k3_df.loc[clustered_k3_df[PRIMARY_CLUSTER_COLUMN] == cluster_id].copy()
        total_events = len(cluster_members)
        partial_accumulator_path = COMPOSITE_EXPORT_DIR / f"k3_cluster_{int(cluster_id)}_field_accumulators.nc"
        partial_box_path = COMPOSITE_EXPORT_DIR / f"k3_cluster_{int(cluster_id)}_event_box_means_partial.csv"
        if not partial_accumulator_path.exists():
            restore_from_drive_cache(partial_accumulator_path)
        if not partial_box_path.exists():
            restore_from_drive_cache(partial_box_path)

        accumulator_ds = None
        processed_events = 0
        cluster_box_records = []
        if partial_accumulator_path.exists() and not FORCE_REBUILD_CLUSTER_COMPOSITES:
            partial_ds = xr.open_dataset(partial_accumulator_path).load()
            cached_total_events = int(partial_ds.attrs.get("total_events", total_events))
            cached_processed_events = int(partial_ds.attrs.get("processed_events", 0))
            if cached_total_events == total_events and cached_processed_events > 0:
                accumulator_ds = partial_ds
                processed_events = min(cached_processed_events, total_events)
                if partial_box_path.exists():
                    cluster_box_records = pd.read_csv(partial_box_path).to_dict("records")
                print(
                    f"Restored partial full-domain composite checkpoint for cluster {cluster_id}: {processed_events}/{total_events} events"
                )
            elif cached_total_events != total_events:
                print(
                    f"Ignoring stale partial checkpoint for cluster {cluster_id}: cached total {cached_total_events} != current total {total_events}"
                )

        if processed_events >= total_events and accumulator_ds is not None:
            cluster_mean_ds, cluster_std_ds, cluster_count_ds = finalize_accumulator_dataset(accumulator_ds)
            cluster_mean_datasets.append(cluster_mean_ds.expand_dims(cluster_id=[cluster_id]))
            cluster_std_datasets.append(cluster_std_ds.expand_dims(cluster_id=[cluster_id]))
            cluster_count_datasets.append(cluster_count_ds.expand_dims(cluster_id=[cluster_id]))
            event_box_records_all.extend(cluster_box_records)
            partial_status_lookup[cluster_id] = {
                "cluster_id": int(cluster_id),
                "cluster_label": CLUSTER_LABELS[int(cluster_id)],
                "processed_events": int(total_events),
                "total_events": int(total_events),
                "complete": True,
            }
            write_partial_status(partial_status_lookup)
            continue

        remaining_members = cluster_members.iloc[processed_events:]
        for offset, (_, row) in enumerate(remaining_members.iterrows(), start=1):
            event_number = processed_events + offset
            event_ds, geometry_925 = compute_event_primary_fields(ds, row, geometry_925=geometry_925)
            if accumulator_ds is None:
                accumulator_ds = initialize_accumulator_dataset(event_ds)
            else:
                accumulator_ds = update_accumulator_dataset(accumulator_ds, event_ds)

            for field_name in PRIMARY_COMPOSITE_FIELDS:
                field = event_ds[field_name]
                for box_name, box in BOXES.items():
                    if box is None:
                        continue
                    cluster_box_records.append(
                        {
                            "cluster_id": int(cluster_id),
                            "cluster_label": CLUSTER_LABELS[int(cluster_id)],
                            "catalog_index": int(row.name),
                            "event_peak": pd.Timestamp(row["event_peak"]),
                            "field": field_name,
                            "field_label": field_display_names[field_name],
                            "box_name": box_name,
                            "box_weighted_mean": lat_weighted_box_mean(field, box),
                        }
                    )

            should_checkpoint = PROGRESS_EVERY > 0 and (event_number % PROGRESS_EVERY == 0 or event_number == total_events)
            if should_checkpoint:
                print(
                    f"Full-domain composite accumulation for cluster {cluster_id}: {event_number}/{total_events} events"
                )
                write_partial_accumulator_checkpoint(
                    accumulator_ds,
                    cluster_id=int(cluster_id),
                    processed_events=int(event_number),
                    total_events=int(total_events),
                    partial_path=partial_accumulator_path,
                )
                pd.DataFrame(cluster_box_records).to_csv(partial_box_path, index=False)
                maybe_copy_to_drive(partial_box_path)
                partial_status_lookup[cluster_id] = {
                    "cluster_id": int(cluster_id),
                    "cluster_label": CLUSTER_LABELS[int(cluster_id)],
                    "processed_events": int(event_number),
                    "total_events": int(total_events),
                    "complete": bool(event_number == total_events),
                }
                write_partial_status(partial_status_lookup)

        if accumulator_ds is None:
            raise RuntimeError(f"No composite fields were accumulated for cluster {cluster_id}")

        write_partial_accumulator_checkpoint(
            accumulator_ds,
            cluster_id=int(cluster_id),
            processed_events=int(total_events),
            total_events=int(total_events),
            partial_path=partial_accumulator_path,
        )
        pd.DataFrame(cluster_box_records).to_csv(partial_box_path, index=False)
        maybe_copy_to_drive(partial_box_path)
        partial_status_lookup[cluster_id] = {
            "cluster_id": int(cluster_id),
            "cluster_label": CLUSTER_LABELS[int(cluster_id)],
            "processed_events": int(total_events),
            "total_events": int(total_events),
            "complete": True,
        }
        write_partial_status(partial_status_lookup)

        cluster_mean_ds, cluster_std_ds, cluster_count_ds = finalize_accumulator_dataset(accumulator_ds)
        cluster_mean_datasets.append(cluster_mean_ds.expand_dims(cluster_id=[cluster_id]))
        cluster_std_datasets.append(cluster_std_ds.expand_dims(cluster_id=[cluster_id]))
        cluster_count_datasets.append(cluster_count_ds.expand_dims(cluster_id=[cluster_id]))
        event_box_records_all.extend(cluster_box_records)

    composite_mean_ds = xr.concat(cluster_mean_datasets, dim="cluster_id")
    composite_std_ds = xr.concat(cluster_std_datasets, dim="cluster_id")
    composite_count_ds = xr.concat(cluster_count_datasets, dim="cluster_id")

    event_box_df = pd.DataFrame(event_box_records_all)
    cluster_box_summary_df = (
        event_box_df.groupby(["cluster_id", "cluster_label", "field", "field_label", "box_name"], dropna=False)["box_weighted_mean"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .reset_index()
        .rename(columns={
            "count": "n_events",
            "mean": "mean_event_box_weighted_mean",
            "median": "median_event_box_weighted_mean",
            "std": "std_event_box_weighted_mean",
            "min": "min_event_box_weighted_mean",
            "max": "max_event_box_weighted_mean",
        })
    )

    composite_map_box_records = []
    for cluster_id in cluster_ids:
        for field_name in PRIMARY_COMPOSITE_FIELDS:
            field = composite_mean_ds[field_name].sel(cluster_id=cluster_id)
            count_field = composite_count_ds[field_name].sel(cluster_id=cluster_id)
            for box_name, box in BOXES.items():
                if box is None:
                    continue
                box_counts = count_field.sel(
                    longitude=slice(box.lon_min, box.lon_max),
                    latitude=slice(box.lat_max, box.lat_min),
                )
                composite_map_box_records.append(
                    {
                        "cluster_id": int(cluster_id),
                        "cluster_label": CLUSTER_LABELS[int(cluster_id)],
                        "field": field_name,
                        "field_label": field_display_names[field_name],
                        "box_name": box_name,
                        "box_weighted_mean_from_composite_map": lat_weighted_box_mean(field, box),
                        "min_gridcell_count_in_box": int(box_counts.min().values),
                        "max_gridcell_count_in_box": int(box_counts.max().values),
                    }
                )
    composite_map_box_df = pd.DataFrame(composite_map_box_records)

    diff_datasets = []
    for left_cluster, right_cluster in [(2, 1), (3, 1), (3, 2)]:
        diff_label = f"cluster_{left_cluster}_minus_{right_cluster}"
        diff_ds = (composite_mean_ds.sel(cluster_id=left_cluster) - composite_mean_ds.sel(cluster_id=right_cluster)).expand_dims(pair_label=[diff_label])
        diff_datasets.append(diff_ds)
    composite_diff_ds = xr.concat(diff_datasets, dim="pair_label")

    composite_mean_ds.to_netcdf(COMPOSITE_MEAN_PATH)
    composite_std_ds.to_netcdf(COMPOSITE_STD_PATH)
    composite_count_ds.to_netcdf(COMPOSITE_COUNT_PATH)
    composite_diff_ds.to_netcdf(COMPOSITE_DIFF_PATH)
    event_box_df.to_csv(COMPOSITE_EVENT_BOX_PATH, index=False)
    cluster_box_summary_df.to_csv(COMPOSITE_BOX_SUMMARY_PATH, index=False)
    composite_map_box_df.to_csv(COMPOSITE_MAP_BOX_PATH, index=False)

    for path in [
        COMPOSITE_MEAN_PATH,
        COMPOSITE_STD_PATH,
        COMPOSITE_COUNT_PATH,
        COMPOSITE_DIFF_PATH,
        COMPOSITE_EVENT_BOX_PATH,
        COMPOSITE_BOX_SUMMARY_PATH,
        COMPOSITE_MAP_BOX_PATH,
    ]:
        maybe_copy_to_drive(path)

print("Primary composite mean fields")
display(composite_mean_ds)
print("\nCluster composite box-summary table")
display(cluster_box_summary_df.head(18))


In [ ]:

import cartopy.crs as ccrs
import cartopy.feature as cfeature

field_specs = [
    {
        "field": "convergence_925_peak",
        "title": "925 hPa convergence composite",
        "subtitle": "Full-domain pointwise mean of 1e5 * max(-(du/dx + dv/dy), 0) at event peak (t0)",
        "cmap": "YlOrRd",
        "extend": "max",
        "cbar": "Convergence [1e-5 s^-1]",
    },
    {
        "field": "z850_anomaly_min_tminus12_to_tplus12",
        "title": "850 hPa geopotential-height anomaly composite",
        "subtitle": "Full-domain pointwise mean of the event-level minimum anomaly over t-12, t0, and t+12",
        "cmap": "RdBu_r",
        "extend": "both",
        "cbar": "Z850 anomaly [gpm]",
    },
    {
        "field": "temperature_gradient_850_max_tminus12_to_tplus12",
        "title": "850 hPa temperature-gradient composite",
        "subtitle": "Full-domain pointwise mean of the event-level maximum |grad T850| over t-12, t0, and t+12",
        "cmap": "viridis",
        "extend": "max",
        "cbar": "|grad T850| [K (100 km)^-1]",
    },
]


def build_levels(field_name: str):
    if field_name == "convergence_925_peak":
        upper = max(6.0, float(composite_mean_ds[field_name].max()) + 0.5)
        return np.arange(0, upper, 0.5)
    if field_name == "z850_anomaly_min_tminus12_to_tplus12":
        max_abs = float(np.nanmax(np.abs(composite_mean_ds[field_name].values)))
        limit = max(180.0, np.ceil(max_abs / 20.0) * 20.0)
        return np.arange(-limit, limit + 20.0, 20.0)
    upper = max(18.0, float(composite_mean_ds[field_name].max()) + 2.0)
    return np.arange(0, upper, 2.0)


for spec in field_specs:
    field_name = spec["field"]
    levels = build_levels(field_name)
    fig, axes = plt.subplots(
        1,
        len(cluster_ids),
        figsize=(5.6 * len(cluster_ids), 5.1),
        subplot_kw={"projection": ccrs.PlateCarree()},
    )
    if len(cluster_ids) == 1:
        axes = [axes]

    mappable = None
    mask_notes = []
    for ax, cluster_id in zip(axes, cluster_ids):
        cluster_event_total = int(cluster_event_counts[int(cluster_id)])
        minimum_count_to_plot = max(1, int(np.ceil(cluster_event_total * MIN_FRACTION_EVENTS_TO_PLOT)))
        field = composite_mean_ds[field_name].sel(cluster_id=cluster_id)
        count_field = composite_count_ds[field_name].sel(cluster_id=cluster_id)
        plot_field = field.where(count_field >= minimum_count_to_plot)

        ax.set_extent(
            [OBJECTIVE_SUBTYPE_DOMAIN.lon_min, OBJECTIVE_SUBTYPE_DOMAIN.lon_max, OBJECTIVE_SUBTYPE_DOMAIN.lat_min, OBJECTIVE_SUBTYPE_DOMAIN.lat_max],
            crs=ccrs.PlateCarree(),
        )
        ax.coastlines(resolution="50m", linewidth=0.9)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5)
        ax.add_feature(cfeature.LAND, facecolor="#f2f2f2", alpha=0.6)

        mappable = ax.contourf(
            plot_field.longitude,
            plot_field.latitude,
            plot_field,
            levels=levels,
            cmap=spec["cmap"],
            extend=spec["extend"],
            transform=ccrs.PlateCarree(),
        )

        gl = ax.gridlines(draw_labels=True, linewidth=0.25, alpha=0.35)
        gl.top_labels = False
        gl.right_labels = False
        if ax is not axes[0]:
            gl.left_labels = False

        cluster_name = CLUSTER_LABELS[int(cluster_id)].split(": ", 1)[1]
        ax.set_title(
            f"Cluster {int(cluster_id)}\n"
            f"{cluster_name}\n"
            f"n={cluster_event_total}; keep grid cells with count >= {minimum_count_to_plot}",
            fontsize=10,
        )
        mask_notes.append(f"C{int(cluster_id)}: count >= {minimum_count_to_plot}")

    fig.suptitle(
        spec["title"] + "\n" + spec["subtitle"],
        y=0.98,
        fontsize=13,
    )
    cbar = fig.colorbar(
        mappable,
        ax=axes,
        orientation="horizontal",
        fraction=0.05,
        pad=0.08,
        aspect=40,
    )
    cbar.set_label(spec["cbar"])
    fig.text(
        0.5,
        0.06,
        f"Units: {spec['cbar']} | Display mask: " + "; ".join(mask_notes),
        ha="center",
        va="center",
        fontsize=9,
    )
    fig.subplots_adjust(top=0.84, bottom=0.22, wspace=0.08)
    output_path = PLOT_DIR / f"k3_primary_composite_{field_name}.png"
    if SAVE_PLOTS:
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        maybe_copy_to_drive(output_path)
    plt.show()


In [ ]:

sample_count_rows = []
for cluster_id in cluster_ids:
    cluster_event_total = int(cluster_event_counts[int(cluster_id)])
    for field_name in PRIMARY_COMPOSITE_FIELDS:
        count_field = composite_count_ds[field_name].sel(cluster_id=cluster_id)
        sample_count_rows.append(
            {
                "cluster_id": int(cluster_id),
                "cluster_label": CLUSTER_LABELS[int(cluster_id)],
                "field": field_name,
                "n_events_in_cluster": cluster_event_total,
                "min_gridcell_count": int(count_field.min().values),
                "median_gridcell_count": float(count_field.median().values),
                "max_gridcell_count": int(count_field.max().values),
            }
        )
sample_count_summary_df = pd.DataFrame(sample_count_rows)
sample_count_summary_path = COMPOSITE_EXPORT_DIR / "k3_cluster_sample_count_summary.csv"
sample_count_summary_df.to_csv(sample_count_summary_path, index=False)
maybe_copy_to_drive(sample_count_summary_path)

pair_to_plot = "cluster_3_minus_2"
if pair_to_plot in composite_diff_ds["pair_label"].values.tolist():
    diff_specs = [
        ("convergence_925_peak", "925 hPa convergence difference", "RdBu_r", np.arange(-3.0, 3.1, 0.25), "Convergence diff [1e-5 s^-1]"),
        ("z850_anomaly_min_tminus12_to_tplus12", "850 hPa z anomaly difference", "RdBu_r", np.arange(-120, 121, 10), "Z850 anomaly diff [gpm]"),
        ("temperature_gradient_850_max_tminus12_to_tplus12", "850 hPa |grad T| difference", "RdBu_r", np.arange(-12, 12.1, 1), "|grad T850| diff [K (100 km)^-1]"),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8), subplot_kw={"projection": ccrs.PlateCarree()})
    for ax, (field_name, title, cmap, levels, cbar_label) in zip(axes, diff_specs):
        ax.set_extent(
            [OBJECTIVE_SUBTYPE_DOMAIN.lon_min, OBJECTIVE_SUBTYPE_DOMAIN.lon_max, OBJECTIVE_SUBTYPE_DOMAIN.lat_min, OBJECTIVE_SUBTYPE_DOMAIN.lat_max],
            crs=ccrs.PlateCarree(),
        )
        ax.coastlines(resolution="50m", linewidth=0.9)
        ax.add_feature(cfeature.BORDERS, linewidth=0.4, alpha=0.5)
        ax.add_feature(cfeature.LAND, facecolor="#f2f2f2", alpha=0.6)
        field = composite_diff_ds[field_name].sel(pair_label=pair_to_plot)
        cf = ax.contourf(
            field.longitude,
            field.latitude,
            field,
            levels=levels,
            cmap=cmap,
            extend="both",
            transform=ccrs.PlateCarree(),
        )
        gl = ax.gridlines(draw_labels=True, linewidth=0.25, alpha=0.35)
        gl.top_labels = False
        gl.right_labels = False
        if ax is not axes[0]:
            gl.left_labels = False
        ax.set_title(title, fontsize=10)
        cbar = fig.colorbar(cf, ax=ax, orientation="horizontal", fraction=0.05, pad=0.08)
        cbar.set_label(cbar_label)
    fig.suptitle(
        "Cluster 3 minus Cluster 2 primary-field differences\n"
        "Use these to see whether the strongest synoptic subtype is spatially different or mostly stronger in amplitude.",
        y=0.98,
        fontsize=13,
    )
    fig.subplots_adjust(top=0.82, bottom=0.16, wspace=0.14)
    difference_plot_path = PLOT_DIR / "k3_cluster3_minus_cluster2_primary_differences.png"
    if SAVE_PLOTS:
        fig.savefig(difference_plot_path, dpi=180, bbox_inches="tight")
        maybe_copy_to_drive(difference_plot_path)
    plt.show()

print("Grid-cell sample-count summary")
display(sample_count_summary_df)
print("\nEvent-level box-mean summary by cluster")
display(cluster_box_summary_df)
print("\nComposite-map box means")
display(composite_map_box_df)
print("\nPairwise cluster checks from Notebook 09 for k=3, cluster 2 vs 3")
display(pairwise_k3_df)


In [ ]:

summary_rows = []
for cluster_id in cluster_ids:
    subset = clustered_k3_df.loc[clustered_k3_df[PRIMARY_CLUSTER_COLUMN] == cluster_id].copy()
    summary_rows.append(
        {
            "cluster_id": int(cluster_id),
            "cluster_label": CLUSTER_LABELS[int(cluster_id)],
            "n_events": int(len(subset)),
            "median_coastal_ratio": float(subset["coastal_to_jpcz_mean_convergence_ratio"].median()),
            "median_hokkaido_z850_anomaly": float(subset["hokkaido_min_z850_anomaly_tminus12_to_tplus12"].median()),
            "median_frontality": float(subset["front_box_max_temp_gradient_850_tminus12_to_tplus12"].median()),
            "median_soj_vorticity": float(subset["sea_of_japan_mean_vorticity_peak_925"].median()),
            "representative_catalog_indices": ", ".join(
                str(int(value))
                for value in representative_events_df.loc[
                    representative_events_df["cluster_id"] == cluster_id,
                    "catalog_index",
                ].head(3)
            ),
        }
    )

cluster_story_df = pd.DataFrame(summary_rows)
cluster_story_path = COMPOSITE_EXPORT_DIR / "k3_cluster_story_summary.csv"
cluster_story_df.to_csv(cluster_story_path, index=False)
maybe_copy_to_drive(cluster_story_path)

print("k=3 interpretation summary")
display(cluster_story_df)
print("\nNotebook 10 now does the following:")
print("- uses the validated k=3 cluster assignments as the working subtype framework")
print("- composites full-domain convergence, z850 anomaly, and temperature-gradient fields rather than only box scalars")
print("- saves pointwise mean, pointwise standard deviation, and pointwise sample-count datasets")
print("- saves event-level and cluster-level box-average summaries for direct comparison with the gridded maps")
print("- keeps the Cluster 3 minus Cluster 2 difference maps available for the strongest subtype contrast")
